# THỰC HÀNH BUỔI 6: SILVER LAYER - DATA CLEANING, JOINS, SHUFFLE & PARQUET

Chào mừng bạn đến với buổi thực hành số 6! Trong buổi này, chúng ta sẽ xây dựng tầng **Silver Layer** trong kiến trúc Lakehouse. Nhiệm vụ chính bao gồm:
- **Data Cleaning**: Làm sạch, loại bỏ trùng lặp, chuẩn hóa kiểu dữ liệu cho 3 bảng: `Orders`, `Customers`, `Order Items`.
- **Timestamp Handling**: Đưa các trường ngày tháng từ dạng String sang Timestamp chuẩn.
- **Join & Shuffle**: Tìm hiểu cơ chế Shuffle trong Spark và cách nó ảnh hưởng tới hiệu năng khi thực hiện phép Join.
- **Broadcast Join**: Sử dụng kỹ thuật tối ưu hóa Map-side Join để tránh Shuffle mạng.
- **Parquet Format**: Ghi dữ liệu sạch xuống tầng Silver ở định dạng lưu trữ dạng cột (Columnar format) tối ưu nhất.

## 1. Khởi tạo Spark Session

Chúng ta sẽ khởi tạo Spark Session ở chế độ `local[*]`. Cấu hình này hỗ trợ cả:
- Đọc/ghi dữ liệu cục bộ từ thư mục `data/raw` và `data/silver`.
- Kết nối đến MinIO (S3 API) qua Docker container mạng nếu Docker đang chạy.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

# Khởi tạo Spark Session với cấu hình hỗ trợ kết nối S3A (MinIO)
spark = SparkSession.builder \
    .appName("Ecommerce Silver Layer Processing") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

# Tắt bớt log không cần thiết
spark.sparkContext.setLogLevel("WARN")
print("Spark Session đã khởi tạo thành công!")

## 2. Thiết lập đường dẫn dữ liệu (Fallback Mode)

Để đảm bảo code có thể chạy ngay cả khi bạn chưa bật Docker / MinIO, chúng ta sẽ cấu hình biến `USE_S3`:
- Nếu `USE_S3 = True`: Đọc và ghi dữ liệu từ MinIO S3 bucket (`s3a://...`).
- Nếu `USE_S3 = False`: Đọc từ `data/raw/*.csv` cục bộ và ghi ra `data/silver/` cục bộ.

In [ ]:
# Đổi thành True nếu Docker / MinIO đang chạy và bạn đã upload dữ liệu lên bucket bronze-layer
USE_S3 = False 

if USE_S3:
    print("Đang cấu hình đọc ghi dữ liệu với MinIO S3...")
    path_orders = "s3a://bronze-layer/olist_orders_dataset.csv"
    path_customers = "s3a://bronze-layer/olist_customers_dataset.csv"
    path_items = "s3a://bronze-layer/olist_order_items_dataset.csv"
    
    silver_orders_out = "s3a://silver-layer/orders"
    silver_customers_out = "s3a://silver-layer/customers"
    silver_items_out = "s3a://silver-layer/order_items"
else:
    print("Đang cấu hình đọc ghi dữ liệu từ thư mục Cục bộ (Local Fallback)...")
    path_orders = os.path.join("..", "data", "raw", "olist_orders_dataset.csv")
    path_customers = os.path.join("..", "data", "raw", "olist_customers_dataset.csv")
    path_items = os.path.join("..", "data", "raw", "olist_order_items_dataset.csv")
    
    silver_orders_out = os.path.join("..", "data", "silver", "orders")
    silver_customers_out = os.path.join("..", "data", "silver", "customers")
    silver_items_out = os.path.join("..", "data", "silver", "order_items")

## 3. Làm sạch dữ liệu Orders (Đơn hàng)

Các bước xử lý:
1. Đọc file Bronze.
2. Chọn lọc các cột cần thiết và đổi tên cột theo quy tắc thống nhất (`snake_case`).
3. Ép kiểu các trường ngày tháng dạng String sang `TimestampType`.
4. Loại bỏ trùng lặp dựa trên khóa chính `order_id` bằng `.dropDuplicates()`.

In [ ]:
print("--- Đang làm sạch tập Orders ---")
df_orders_raw = spark.read.csv(path_orders, header=True, inferSchema=True)

df_orders_clean = df_orders_raw.select(
    col("order_id"),
    col("customer_id"),
    col("order_status").alias("status"),
    col("order_purchase_timestamp").cast("timestamp").alias("purchase_timestamp"),
    col("order_approved_at").cast("timestamp").alias("approved_at"),
    col("order_delivered_carrier_date").cast("timestamp").alias("delivered_carrier_date"),
    col("order_delivered_customer_date").cast("timestamp").alias("delivered_customer_date"),
    col("order_estimated_delivery_date").cast("timestamp").alias("estimated_delivery_date")
).dropDuplicates(["order_id"])

print(f"[Orders] Số dòng trước làm sạch: {df_orders_raw.count()}")
print(f"[Orders] Số dòng sau làm sạch: {df_orders_clean.count()}")
df_orders_clean.printSchema()
df_orders_clean.show(5, truncate=False)

## 4. Làm sạch dữ liệu Customers (Khách hàng)

Các bước xử lý:
1. Đọc file Bronze.
2. Chọn lọc các cột cần thiết.
3. Loại bỏ trùng lặp dựa trên khóa chính `customer_id`.

In [ ]:
print("--- Đang làm sạch tập Customers ---")
df_customers_raw = spark.read.csv(path_customers, header=True, inferSchema=True)

df_customers_clean = df_customers_raw.select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix").cast("int").alias("zip_code_prefix"),
    col("customer_city").alias("city"),
    col("customer_state").alias("state")
).dropDuplicates(["customer_id"])

print(f"[Customers] Số dòng trước làm sạch: {df_customers_raw.count()}")
print(f"[Customers] Số dòng sau làm sạch: {df_customers_clean.count()}")
df_customers_clean.printSchema()
df_customers_clean.show(5, truncate=False)

## 5. Làm sạch dữ liệu Order Items (Chi tiết đơn hàng)

Các bước xử lý:
1. Đọc file Bronze.
2. Ép kiểu giá tiền (`price`) và phí vận chuyển (`freight_value`) sang `double`.
3. Ép kiểu hạn giao hàng (`shipping_limit_date`) sang `timestamp`.
4. Loại bỏ trùng lặp dựa trên cặp khóa chính `(order_id, order_item_id)`.

In [ ]:
print("--- Đang làm sạch tập Order Items ---")
df_items_raw = spark.read.csv(path_items, header=True, inferSchema=True)

df_items_clean = df_items_raw.select(
    col("order_id"),
    col("order_item_id").cast("int"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date").cast("timestamp").alias("shipping_limit_date"),
    col("price").cast("double"),
    col("freight_value").cast("double")
).dropDuplicates(["order_id", "order_item_id"])

print(f"[Order Items] Số dòng trước làm sạch: {df_items_raw.count()}")
print(f"[Order Items] Số dòng sau làm sạch: {df_items_clean.count()}")
df_items_clean.printSchema()
df_items_clean.show(5, truncate=False)

## 6. Tìm hiểu về Join, Shuffle & Broadcast Join

### Khái niệm Shuffle trong Spark:
- **Shuffle** là quá trình phân phối lại dữ liệu giữa các phân vùng (partitions) trên toàn mạng lưới (giữa các Executors/Workers).
- Phép Join mặc định trong Spark (Sort Merge Join) yêu cầu hai bảng phải được băm (hash) và gom nhóm theo khóa Join. Điều này gây ra quá trình Shuffle cực kỳ tốn I/O mạng, đĩa và CPU.

### Giải pháp tối ưu: Broadcast Join (Map-side Join)
- Nếu một trong hai bảng có kích thước nhỏ (ví dụ bảng `Customers` so với bảng `Orders`), Spark sẽ sao chép toàn bộ bảng nhỏ này đến tất cả các Executors.
- Khi đó, Executors chỉ cần thực hiện phép Join cục bộ ngay trên phân vùng của bảng lớn mà không cần chuyển đổi dữ liệu của bảng lớn qua mạng.
- Giúp cải thiện tốc độ đáng kể cho các phép Join lớn-nhỏ.

### 6.1 Thực hiện Join mặc định (Sort Merge Join)

Chúng ta sẽ thực hiện join `orders` với `customers` dựa trên `customer_id` theo cách mặc định. 
Sau đó sử dụng `.explain(True)` để quan sát xem Spark có sinh ra toán tử `Exchange` (chính là biểu thị cho bước Shuffle) hay không.

In [ ]:
# Tạm thời tắt cơ chế tự động broadcast của Spark để mô phỏng SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

df_joined_standard = df_orders_clean.join(df_customers_clean, on="customer_id", how="inner")

# In Execution Plan của Standard Join
df_joined_standard.explain(True)

### 6.2 Thực hiện Broadcast Join (Tối ưu hóa)

Bây giờ chúng ta sẽ ép buộc Spark thực hiện Broadcast Join bằng cách sử dụng hàm `broadcast()` bao bọc bảng `customers` nhỏ hơn. 
Chúng ta hãy quan sát Physical Plan bằng `.explain(True)` để thấy sự khác biệt!

In [ ]:
# Sử dụng hàm broadcast cho bảng nhỏ hơn (Customers)
df_joined_broadcast = df_orders_clean.join(broadcast(df_customers_clean), on="customer_id", how="inner")

# In Execution Plan của Broadcast Join
df_joined_broadcast.explain(True)

### NHẬN XÉT SỰ KHÁC BIỆT TRONG EXECUTION PLAN:
1. **Mặc định (Sort Merge Join):**
   - Bạn sẽ thấy toán tử `Exchange hashpartitioning(...)` xuất hiện ở cả hai nhánh dữ liệu trước khi thực hiện `SortMergeJoin`. Đây chính là bước Shuffle dữ liệu qua mạng.
2. **Tối ưu (Broadcast Join):**
   - Xuất hiện toán tử `BroadcastExchange` ở nhánh của bảng nhỏ (`Customers`). Nhánh của bảng lớn (`Orders`) hoàn toàn không có `Exchange` nào! Phép join được nâng cấp thành `BroadcastHashJoin` chạy trực tiếp rất nhanh.

## 7. Ghi định dạng Parquet xuống Silver Layer

**Lợi ích của định dạng Parquet:**
- Lưu trữ dạng cột (**Columnar Storage**): giúp Spark chỉ đọc đúng những cột cần dùng khi phân tích, tối ưu hóa I/O.
- Nén hiệu quả (**High Compression**): Tiết kiệm dung lượng lưu trữ đáng kể so với CSV.
- Lưu trữ metadata và schema: Spark không cần phỏng đoán kiểu dữ liệu (`inferSchema`) khi đọc lại Parquet.

Chúng ta ghi dữ liệu ở chế độ `overwrite` để chạy lại mà không bị lỗi ghi đè dữ liệu cũ.

In [ ]:
print("--- Đang ghi dữ liệu Parquet xuống Silver Layer ---")

# Tạo thư mục cha nếu lưu ở local
if not USE_S3:
    os.makedirs(os.path.join("..", "data", "silver"), exist_ok=True)

df_orders_clean.write.mode("overwrite").parquet(silver_orders_out)
print("-> Đã ghi Orders sạch xuống Silver Layer.")

df_customers_clean.write.mode("overwrite").parquet(silver_customers_out)
print("-> Đã ghi Customers sạch xuống Silver Layer.")

df_items_clean.write.mode("overwrite").parquet(silver_items_out)
print("-> Đã ghi Order Items sạch xuống Silver Layer.")

print("Quá trình ghi Silver Layer hoàn tất thành công!")

## 8. Đọc ngược lại dữ liệu Parquet từ Silver Layer để kiểm tra

Chúng ta đọc lại dữ liệu Parquet vừa lưu để kiểm chứng schema đã được bảo toàn chính xác mà không cần `inferSchema`.

In [ ]:
print("--- Đọc kiểm chứng từ Silver Layer ---")
df_orders_silver = spark.read.parquet(silver_orders_out)

df_orders_silver.printSchema()
print(f"Số dòng của Orders trong Silver Layer: {df_orders_silver.count()}")
df_orders_silver.show(5, truncate=False)

## 9. Dọn dẹp tài nguyên

Dừng Spark Session sau khi hoàn thành.

In [ ]:
spark.stop()
print("Spark Session đã dừng thành công!")